# HyperLogLog

Wiki reference for [HyperLogLog](https://ml-viz-ruby.vercel.app/wiki/hyperloglog).

**The idea in one sentence.** Count distinct elements in a few kilobytes by splitting hashes across m registers, each tracking the longest run of leading zeros, then combining them with a harmonic mean.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)
import hashlib

## 1. From scratch — registers and the harmonic-mean estimator

Use the first `p` bits of a 64-bit hash to pick one of `m = 2^p` registers; in that register keep the max `rho` = position of the leftmost 1 in the remaining bits. Estimate `n̂ = α_m · m² / Σ 2^(-M[j])`, with a linear-counting correction when registers are still empty.

In [ ]:
def hll(items, p=10):
    m = 1 << p
    reg = np.zeros(m, dtype=int)
    bits = 64 - p
    mask = (1 << bits) - 1
    for x in items:
        h = int(hashlib.sha1(str(x).encode()).hexdigest(), 16) & ((1 << 64) - 1)
        j = h >> bits                       # first p bits -> register index
        rem = h & mask                      # remaining bits
        rho = bits - rem.bit_length() + 1 if rem else bits + 1   # pos. of leftmost 1
        if rho > reg[j]:
            reg[j] = rho
    return reg

def estimate(reg):
    m = len(reg)
    alpha = 0.7213 / (1 + 1.079 / m) if m >= 128 else 0.673
    E = alpha * m * m / np.sum(2.0 ** (-reg.astype(float)))
    V = int(np.count_nonzero(reg == 0))
    if E <= 2.5 * m and V > 0:
        E = m * np.log(m / V)               # small-range (linear counting) correction
    return E

## 2. The library way — validate against the exact truth and the error formula

The guarantee is a *relative standard error* of `1.04/√m`. We count a stream with heavy duplication and assert HLL lands within `3σ` of the true distinct count while using only `m` small registers.

In [ ]:
p = 10
m = 1 << p
true_n = 200_000
stream = np.random.randint(0, true_n, size=1_000_000)   # ~5x duplication
reg = hll(stream, p=p)
est = estimate(reg)
rel_err = abs(est - true_n) / true_n
sigma = 1.04 / np.sqrt(m)
print(f'registers m = {m}  (~{m * 6 // 8} bytes)')
print(f'true distinct = {true_n:,}   HLL estimate = {est:,.0f}   rel error = {rel_err:.3%}')
print(f'predicted standard error 1.04/sqrt(m) = {sigma:.3%}')
assert rel_err < 3 * sigma, 'HLL estimate must be within 3 standard errors of the truth'
print('HLL counts 200k distinct from a 1M stream in a few KB, within 3σ ✓')

## 3. Visualize it — accuracy improves as √m

Sweep the register count and plot the relative error; it should shrink like `1.04/√m`.

In [ ]:
ps = [6, 8, 10, 12, 14]
errs, preds = [], []
for pp in ps:
    e = estimate(hll(stream, p=pp))
    errs.append(abs(e - true_n) / true_n)
    preds.append(1.04 / np.sqrt(1 << pp))
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot([1 << pp for pp in ps], errs, 'o-', color='#14b8a6', label='observed rel. error')
ax.plot([1 << pp for pp in ps], preds, '--', color='#e2e8f0', label='1.04/√m')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('registers m'); ax.set_ylabel('relative error (log)')
ax.set_title('HyperLogLog error shrinks like 1/√m', color='white')
ax.legend(); ax.grid(alpha=0.2, which='both'); plt.show()

**What to notice:** each 4× increase in registers roughly halves the error, tracking the `1.04/√m` line — and even `m = 16384` registers is only ~12 KB. Doubling accuracy is cheap because memory is the (tiny) budget you spend to buy precision.

## 4. Gotchas

- **Hash quality is everything** — bits must be uniform; a weak hash biases `rho`.
- **Mergeable by element-wise max** of registers (great for distributed counting and unions); **intersections** need inclusion-exclusion, which amplifies error.
- **Cardinality only** — not membership (Bloom) or frequency (Count-Min).
- Real **HLL++** uses 64-bit hashes, a bias-correction table, and a sparse mode for small counts.

## 5. Your turn

### Exercise — the HLL estimate

Given a filled register array, implement the raw estimator `α_m · m² / Σ 2^(-M[j])` (skip the range corrections here). Use `α_m = 0.7213 / (1 + 1.079/m)` for `m ≥ 128`.

In [ ]:
def hll_estimate_raw(reg):
    m = len(reg)
    alpha = 0.7213 / (1 + 1.079 / m) if m >= 128 else 0.673
    # TODO(you): return alpha * m**2 / sum(2^(-M[j]) over registers)
    return ...


In [ ]:
# Checks — run me
reg_full = hll(stream, p=12)
assert np.count_nonzero(reg_full == 0) == 0, 'registers should be populated at this cardinality'
raw = hll_estimate_raw(reg_full)
assert abs(raw - true_n) / true_n < 0.05, 'raw estimator should be within ~5% at m=4096'
# a uniform all-ones-ish register array must give a large estimate
assert hll_estimate_raw(np.full(256, 5)) > hll_estimate_raw(np.full(256, 1))
print('✅ Exercise passed  (raw estimate = %.0f)' % raw)

<details>
<summary>💡 Show solution</summary>

```python
def hll_estimate_raw(reg):
    m = len(reg)
    alpha = 0.7213 / (1 + 1.079 / m) if m >= 128 else 0.673
    return alpha * m * m / np.sum(2.0 ** (-reg.astype(float)))
```

</details>

## 6. Key takeaways

- HLL = Flajolet-Martin + **stochastic averaging** over `m` registers + a **harmonic-mean** estimate.
- Relative error `≈ 1.04/√m`; a few KB counts billions within a couple percent.
- Registers are **mergeable by max** — ideal for distributed and windowed cardinality.
- Back to [Streaming Algorithms & Sketches](https://ml-viz-ruby.vercel.app/courses/streaming-ml/02-streaming-algorithms).